# Primer análisis

Notebook de arranque del agente de análisis de datos.

El orden importa: **perfilar antes de analizar**. Saltarse el perfil es la
causa más común de conclusiones equivocadas.

1. Cargar el dataset
2. Perfilar la calidad de los datos
3. Responder las preguntas con SQL (DuckDB) o pandas
4. Graficar solo si el gráfico aclara algo

## 1. Preparación

`sys.path` permite importar los módulos de `src/` desde el notebook.

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

from perfilar import cargar, perfilar  # noqa: E402

pd.set_option("display.max_columns", 50)
print(f"Proyecto: {RAIZ}")
print(f"pandas {pd.__version__} | duckdb {duckdb.__version__}")

## 2. Cargar el dataset

Pon tu archivo de Kaggle en `data/raw/` y escribe su nombre aquí.

In [ ]:
NOMBRE_ARCHIVO = "mi_dataset.csv"  # <-- cambia esto

ruta = RAIZ / "data" / "raw" / NOMBRE_ARCHIVO
df = cargar(ruta)
df.head()

## 3. Perfilar

Qué buscar en la salida:

- **Nulos altos** en una columna que necesitas: decide si imputar o descartar.
- **Duplicados**: ¿son errores de carga o registros legítimos repetidos?
- **Fechas como texto**: hay que convertirlas o no se puede ordenar por tiempo.
- **Categorías repetidas** por mayúsculas: agrupar sin normalizar da totales partidos.

In [ ]:
print(perfilar(df, ruta.name))

## 4. Consultar con SQL

DuckDB consulta el DataFrame directamente: no hay que cargar nada a una base
de datos. El nombre de la variable de Python se usa como nombre de tabla.

In [ ]:
# Pregunta descriptiva: totales por categoría.
# Ajusta los nombres de columna a los de tu dataset.
consulta = """
SELECT
    categoria,
    COUNT(*)        AS registros,
    SUM(valor)      AS total,
    AVG(valor)      AS promedio
FROM df
GROUP BY categoria
ORDER BY total DESC
"""

duckdb.sql(consulta).df()

## 5. Graficar

Un gráfico por pregunta, y solo si muestra algo que la tabla no deja ver
(una tendencia, una forma, un valor fuera de lugar).

In [ ]:
resultado = duckdb.sql(consulta).df()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(resultado["categoria"], resultado["total"], color="#0F766E")
ax.invert_yaxis()
ax.set_xlabel("Total")
ax.set_title("Total por categoría")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

(RAIZ / "reports").mkdir(exist_ok=True)
fig.savefig(RAIZ / "reports" / "total_por_categoria.png", dpi=150)
plt.show()

## 6. Conclusión

Todo análisis cierra con estas tres líneas:

- **Qué encontré:**
- **Qué significa:**
- **Qué revisaría después:**